# Newton's law of cooling

This notebook connects the reusable PINN core to the first physical problem. The governing equation is:

$$\frac{dT}{dt}=-k(T-T_{ambient}).$$

The model is trained from the residual and the initial condition, not from the analytical temperature curve.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Using project root: {PROJECT_ROOT}")


Using project root: /Users/nitinbhardwaj2006gmail.com/Desktop/physics-informed-neural-networks


## Why normalize temperature?

The code predicts:

$$y(t)=\frac{T(t)-T_{ambient}}{T_0-T_{ambient}}.$$

That means `y(0)=1` and the ambient state is `y=0`. The residual becomes `dy/dt + k*y`, which is numerically better conditioned than predicting temperatures around 90 directly.


In [2]:
import numpy as np
from problems.odes import CoolingConfig, cooling_exact, cooling_to_temperature

config = CoolingConfig()
time = np.linspace(0.0, config.final_time, 5)
print("Time:", time)
print("Analytical temperature (evaluation only):", cooling_exact(time, config))
print("Normalized [1, 0] in degrees:", cooling_to_temperature(np.array([1.0, 0.0]), config))


Time: [ 0.   2.5  5.   7.5 10. ]
Analytical temperature (evaluation only): [90.         49.18034138 32.16417604 25.07078299 22.11381684]
Normalized [1, 0] in degrees: [90. 20.]


## Run the PINN experiment

Run the project script from the repository root:

```bash
python experiments/train_cooling.py --epochs 1500
```

The script creates `results/cooling_results.png` and `results/cooling_metrics.json`. The plot compares the trained PINN with the exact curve after training.


In [3]:
# Optional: run a shorter experiment from a notebook.
# import subprocess
# subprocess.run(["python", "experiments/train_cooling.py", "--epochs", "500"], cwd=PROJECT_ROOT)


## What to check

1. `cooling_residual` uses a derivative of the network output.
2. `cooling_initial_condition` targets 1.0 because the state is normalized.
3. `cooling_exact` is used for evaluation, not training.
4. RMSE answers a different question from the training loss.
